# Sentiment Analysis

## Dev. Notes

`argilla` is being executed locally using `docker compose` by executing the command:

```
docker compose -f docker/docker-compose-argilla.yml
```

or by executing the script:

```
./scripts/argilla-up.sh
```

### `argilla` credendials in the UI
User: argilla
Pwd: 12345678


## Preliminaries


### Datasets

In [1]:
from datasets import load_dataset

imdb_ds = load_dataset("stanfordnlp/imdb")

to_label1, to_label2 = imdb_ds['train'].train_test_split(test_size=0.5, seed=42).values()

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

### Model

In [2]:
from transformers import pipeline

sentiment_classifier = pipeline(
    model="distilbert-base-uncased-finetuned-sst-2-english",
    task="sentiment-analysis",
    return_all_scores=True,
)

to_label1[3]['text'], sentiment_classifier(to_label1[3]['text'])

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

("This is a fantastic series first and foremost. It is very well done and very interesting. As a huge WWII buff, I had learned a lot before seeing this series. One of the best things this has going for it is all the interviews with past individuals back when the war was relatively fresh in their minds, comparatively speaking that is. It is nothing against the men that you see getting interviewed in the programs of today, it is just that most of these men weren't really involved in the upper echelons of what was happening then. One of the best parts is the narrating by Sir Laurence Oliver. I would recommend this to anyone that wants to learn about WWII, but really think only the die-hards (such as myself) will want to buy this or watch it more than once. My only real complaint about this entire series is that some of the facts aren't quite as accurate as we now know. Especially with the information about Soviet Union is exaggerated or just plain inaccurate in places. That information is

### Init `argilla` client

In [6]:
import argilla as rg

client = rg.Argilla(
    api_url="http://localhost:6900",
    api_key="argilla.apikey"
)

client.me

User(id=UUID('1ff47bd9-37eb-4ad0-85bf-8fd9b8048e58') inserted_at=datetime.datetime(2026, 4, 17, 20, 39, 37, 530814) updated_at=datetime.datetime(2026, 4, 17, 20, 39, 37, 530814) username='argilla' role=<Role.owner: 'owner'> first_name='argilla' last_name=None password=None)

## 1. Run the pre-trained model

### Predict

In [3]:
def predict(examples):
    return {"predictions": sentiment_classifier(examples['text'], truncation=True)}

# add .select(range(10)) before map if you just want to test this quickly with 10 examples
to_label1 = to_label1.map(predict, batched=True, batch_size=4)

Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

### Create records, logs

In [15]:
to_label1[0]

{'text': "Set in 2017 (although one might easily mistake it for 1987, judging by the hairstyles and clothing), The Running Man sees all-round good guy Ben Richards (Schwarzeneggar) framed for a crime he didn't commit. After a daring prison break, he is captured and entered as a contestant in the brutal TV game show The Running Man, along with some fellow escapees and the pretty token female, Amber (Maria Conchita Alonso),.<br /><br />Used by the totalitarian government as a way of controlling the masses, the show pits convicts against a range of colourful (and often quite camp) opponents, each having his own unique killing style: Dynamo fires electricity from a special suit, Buzzsaw uses chainsaws, Sub Zero has a razor edged ice hockey stick, and Fireball prefers a flamethrower to finish off contenders. But these killers are no match for Ben Richards, who dispatches each one in a fittingly gruesome manner (followed by the obligatory witticism).<br /><br />Towards the end of the movie, 

In [18]:
import argilla as rg

workspace = "default"
dataset_name = "sentiment-predictions"
question_name = "sentiment"

# Create (or reuse) a Feedback dataset
dataset = client.datasets(name=dataset_name, workspace=workspace)
if dataset is None:
    settings = rg.Settings(
        fields=[
            rg.TextField(name="text"),
        ],
        questions=[
            rg.LabelQuestion(
                name=question_name,
                labels=["NEGATIVE", "POSITIVE"],
            ),
        ],
        metadata=[
            rg.TermsMetadataProperty(name="category"),
        ],
    )
    dataset = rg.Dataset(
        name=dataset_name,
        workspace=workspace,
        settings=settings,
        client=client,
    )
    dataset.create()


def normalize_predictions(predictions):
    """Normalize model outputs to a list of {'label', 'score'} dicts."""
    if isinstance(predictions, dict):
        return [predictions]
    if isinstance(predictions, list):
        if not predictions:
            return []
        # Some pipelines may return [[{...}, {...}]] per example
        if isinstance(predictions[0], list):
            return predictions[0]
        return predictions
    return []


# Build records from your model outputs
records = []
for example in to_label1.shuffle():
    preds = normalize_predictions(example["predictions"])
    if not preds:
        continue

    top_pred = max(preds, key=lambda p: p["score"])

    records.append(
        rg.Record(
            fields={"text": example["text"]},
            metadata={"category": str(example["label"])},
            suggestions=[
                rg.Suggestion(
                    question_name=question_name,
                    value=top_pred["label"],
                    score=float(top_pred["score"]),
                    agent="distilbert-base-uncased-finetuned-sst-2-english",
                )
            ],
        )
    )

# Log records
dataset.records.log(records)
print(f"Logged {len(records)} records")

Sending records...: 49batch [02:20,  2.88s/batch]                     

Logged 12500 records


## 2. Explore and Label data using the pre-trained model

![Labeling data](../images/label-data.png)

## 3. Fine-tune the pre-trained model

In [102]:
import pandas as pd

pre_trained_ds = client.datasets(name="sentiment-predictions", workspace="default")


In [103]:
records = pre_trained_ds.records.to_list(flatten=True)

df = pd.DataFrame(records)
validated_df = df[df["sentiment.responses.status"].apply(lambda s: s is not None and "submitted" in s)]

validated_df.head()


,id,status,_server_id,text,category,sentiment.suggestion,sentiment.suggestion.score,sentiment.suggestion.agent,sentiment.responses.users,sentiment.responses,sentiment.responses.status
10,6f7e7787-5e6e-4ff2-b06b-7cbf8beb6764,completed,02bb1d7e-fb74-4951-8d12-d70714e3b68f,"Usually, I don't think Hollywood productions a...",0,NEGATIVE,0.996319,distilbert-base-uncased-finetuned-sst-2-english,[1ff47bd9-37eb-4ad0-85bf-8fd9b8048e58],[NEGATIVE],[submitted]
53,f9ed939a-13d2-4307-8399-74dfa2a8a9f9,completed,6ddba8b3-63bc-4278-82ad-c18e13cb311a,"I have watched 3 episodes of Caveman, and I ha...",0,NEGATIVE,0.999227,distilbert-base-uncased-finetuned-sst-2-english,[1ff47bd9-37eb-4ad0-85bf-8fd9b8048e58],[NEGATIVE],[submitted]
186,1c263576-266a-4b5c-9c98-7dde024b7c97,completed,6253efbf-b4cf-437b-9929-d6ca479d69e7,This movie is by far the worst movie ever made...,0,NEGATIVE,0.999787,distilbert-base-uncased-finetuned-sst-2-english,[1ff47bd9-37eb-4ad0-85bf-8fd9b8048e58],[NEGATIVE],[submitted]
384,cb167b89-10fc-4ac9-99fd-35e85feb3bf2,completed,456a543d-9261-4216-a0cf-5d0aca0b4afb,This movie is proof that film noire is an endu...,1,NEGATIVE,0.966583,distilbert-base-uncased-finetuned-sst-2-english,[1ff47bd9-37eb-4ad0-85bf-8fd9b8048e58],[NEGATIVE],[submitted]
399,5c4bab98-b132-4dbc-bea7-715429e62f63,completed,6f482b4c-a31c-4a1a-a4e6-06e76049005a,And I mean ultra light. This film features fou...,0,NEGATIVE,0.999575,distilbert-base-uncased-finetuned-sst-2-english,[1ff47bd9-37eb-4ad0-85bf-8fd9b8048e58],[NEGATIVE],[submitted]


### Prepare training and test datasets

In [104]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

question_name = "sentiment"
status_col = f"{question_name}.responses.status"
value_col = f"{question_name}.responses"

# If you have not filtered yet, keep only validated (submitted) annotations
# work_df = df[df[status_col].apply(lambda s: isinstance(s, list) and "submitted" in s)].copy()

# If you have not filtered yet, keep only validated (submitted) annotations
work_df = validated_df.copy()

# Extract one label per row from Argilla response payload
def extract_label(x):
    # Typical shapes: ["POSITIVE"], ["NEGATIVE"], or plain string
    if isinstance(x, list):
        return x[0] if len(x) > 0 else None
    if isinstance(x, str):
        return x
    return None

work_df["label_text"] = work_df[value_col].apply(extract_label)

# Keep only rows usable for training
work_df = work_df.dropna(subset=["text", "label_text"]).copy()
work_df = work_df[work_df["label_text"].isin(["NEGATIVE", "POSITIVE"])].copy()

# Encode labels to integers
label2id = {"NEGATIVE": 0, "POSITIVE": 1}
id2label = {v: k for k, v in label2id.items()}
work_df["label"] = work_df["label_text"].map(label2id).astype("int64")

# Optional cleanup
work_df = work_df.drop_duplicates(subset=["text", "label"])

# Train/validation split (stratified)
train_df, val_df = train_test_split(
    work_df[["text", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=work_df["label"]
)

# Convert to Hugging Face datasets
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))
ds = DatasetDict({"train": train_ds, "validation": val_ds})

print(ds)
print("Label mapping:", label2id)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 169
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 43
    })
})
Label mapping: {'NEGATIVE': 0, 'POSITIVE': 1}


In [63]:
from transformers import AutoTokenizer

# tokenize our datasets
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train_ds = train_ds.map(tokenize_function, batched=True)
tokenized_val_ds = val_ds.map(tokenize_function, batched=True)

Map:   0%|          | 0/169 [00:00<?, ? examples/s]

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Configure the **Trainer**

In [71]:
import numpy as np
from evaluate import load
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english",
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir="distilbert-base-uncased-sentiment-imdb-reviews",
    eval_strategy="epoch",  # Transformers v5+: renamed from evaluation_strategy
    save_strategy="epoch",
    logging_steps=30,
    do_train=True,
    do_eval=True,
)

metric = load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


trainer = Trainer(
    args=training_args,
    model=model,
    train_dataset=tokenized_train_ds,
    eval_dataset=tokenized_val_ds,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### Train the model...

In [72]:
trainer.train()


/home/fvcg/projects/uni-ai/.ai-env/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.168349,0.953488
2,0.476841,0.123112,0.976744
3,0.026320,0.219407,0.953488


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/fvcg/projects/uni-ai/.ai-env/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/fvcg/projects/uni-ai/.ai-env/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=66, training_loss=0.22889968043080333, metrics={'train_runtime': 605.0512, 'train_samples_per_second': 0.838, 'train_steps_per_second': 0.109, 'total_flos': 67160971118592.0, 'train_loss': 0.22889968043080333, 'epoch': 3.0})

## 4. Testing the fine-tuned model

In [73]:
finetuned_sentiment_classifier = pipeline(
    model=model.to("cpu"),
    tokenizer=tokenizer,
    task="sentiment-analysis",
    return_all_scores=True
)

In [75]:
finetuned_sentiment_classifier(
    "Take none of it seriously. Just go in and have a good time. If this wasn't what high school life was like in the seventies, then it should have been."
), sentiment_classifier(
"Take none of it seriously. Just go in and have a good time. If this wasn't what high school life was like in the seventies, then it should have been.")

([{'label': 'POSITIVE', 'score': 0.9998425245285034}],
 [{'label': 'POSITIVE', 'score': 0.996845543384552}])

In [76]:
finetuned_sentiment_classifier(
    "I don't know how to interpret this movie, its awkward."
), sentiment_classifier(
    "I don't know how to interpret this movie, its awkward."
)

([{'label': 'NEGATIVE', 'score': 0.9997567534446716}],
 [{'label': 'NEGATIVE', 'score': 0.9995488524436951}])

## 5. Run out fine-tuned model over the dataset and log the predictions

In [80]:
model_df = df[df["status"] == "pending"]

model_df.head(1)


,id,status,_server_id,text,category,sentiment.suggestion,sentiment.suggestion.score,sentiment.suggestion.agent,sentiment.responses.users,sentiment.responses,sentiment.responses.status
0,7c52c9cf-363c-4f16-a6b9-03ca36c2b663,pending,ddcc8fe1-ebd5-447e-b1f0-de92aaf58dee,"In film, I feel as though it should be more th...",0,NEGATIVE,0.564229,distilbert-base-uncased-finetuned-sst-2-english,None,None,None


In [98]:
def predictFineTuned(examples):

    return {
        "predictions": finetuned_sentiment_classifier(examples["text"], truncation=True),
        "prediction_agent": ["distilbert-base-uncased-sentiment-imdb-reviews"]
        }

model_ds = Dataset.from_pandas(model_df)
model_ds_predicted = model_ds.map(predictFineTuned)

Map:   0%|          | 0/12288 [00:00<?, ? examples/s]

In [99]:
workspace2 = "default"
dataset_name2 = "sentiment-predictions-fine-tuned"
question_name2 = "sentiment"

# Create (or reuse) a Feedback dataset
dataset2 = client.datasets(name=dataset_name2, workspace=workspace2)
if dataset2 is None:
    settings = rg.Settings(
        fields=[
            rg.TextField(name="text"),
        ],
        questions=[
            rg.LabelQuestion(
                name=question_name2,
                labels=["NEGATIVE", "POSITIVE"],
            ),
        ],
        metadata=[
            rg.TermsMetadataProperty(name="category"),
        ],
    )
    dataset2 = rg.Dataset(
        name=dataset_name2,
        workspace=workspace2,
        settings=settings,
        client=client,
    )
    dataset2.create()



# Build records from your model outputs
records2 = []
for example in to_label1.shuffle():
    preds = normalize_predictions(example["predictions"])
    if not preds:
        continue

    top_pred = max(preds, key=lambda p: p["score"])

    records2.append(
        rg.Record(
            fields={"text": example["text"]},
            metadata={"category": str(example["label"])},
            suggestions=[
                rg.Suggestion(
                    question_name=question_name2,
                    value=top_pred["label"],
                    score=float(top_pred["score"]),
                    agent="distilbert-base-uncased-finetuned-sst-2-english",
                )
            ],
        )
    )

# Log records
dataset2.records.log(records2)
print(f"Logged {len(records2)} records")

/home/fvcg/projects/uni-ai/.ai-env/lib/python3.14/site-packages/argilla/client.py:356: UserWarning: Dataset with name 'sentiment-predictions-fine-tuned' not found in workspace 'default'
  warnings.warn(f"Dataset with name {name!r} not found in workspace {workspace.name!r}")
Sending records...: 49batch [01:43,  2.12s/batch]                     

Logged 12500 records


## 6. Explore and label data with the fine-tuned model

![Labeled Data Fine Tunned](../images/label-data-fine-tunned.png)

## 7. Fine-tuning with the extended training dataset

In [105]:
import pandas as pd

fine_tuned_ds = client.datasets(name="sentiment-predictions-fine-tuned", workspace="default")

records_fine_tuned = fine_tuned_ds.records.to_list(flatten=True)

df_fine = pd.DataFrame(records_fine_tuned)


In [106]:
question_name = "sentiment"
status_col = f"{question_name}.responses.status"
value_col = f"{question_name}.responses"

work_df2 = df_fine.copy()

# Extract one label per row from Argilla response payload
def extract_label(x):
    # Typical shapes: ["POSITIVE"], ["NEGATIVE"], or plain string
    if isinstance(x, list):
        return x[0] if len(x) > 0 else None
    if isinstance(x, str):
        return x
    return None

work_df2["label_text"] = work_df2[value_col].apply(extract_label)

# Keep only rows usable for training
work_df2 = work_df2.dropna(subset=["text", "label_text"]).copy()
work_df2 = work_df2[work_df2["label_text"].isin(["NEGATIVE", "POSITIVE"])].copy()

# Encode labels to integers
label2id2 = {"NEGATIVE": 0, "POSITIVE": 1}
id2label2 = {v: k for k, v in label2id2.items()}
work_df2["label"] = work_df2["label_text"].map(label2id2).astype("int64")

# Optional cleanup
work_df2 = work_df2.drop_duplicates(subset=["text", "label"])

# Train/validation split (stratified)
train_df2, val_df2 = train_test_split(
    work_df2[["text", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=work_df2["label"]
)

# Convert to Hugging Face datasets
train_ds2 = Dataset.from_pandas(train_df2.reset_index(drop=True))
val_ds2 = Dataset.from_pandas(val_df2.reset_index(drop=True))
ds2 = DatasetDict({"train": train_ds2, "validation": val_ds2})

print(ds2)
print("Label mapping:", label2id2)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 80
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 20
    })
})
Label mapping: {'NEGATIVE': 0, 'POSITIVE': 1}


In [107]:
from datasets import concatenate_datasets

train_dataset_concatenated = concatenate_datasets([train_ds, train_ds2])

In [108]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [114]:
new_train_ds = train_dataset_concatenated.shuffle(seed=42)

# Tokenize the concatenated dataset and validation dataset for Trainer
new_train_ds_tok = new_train_ds.map(tokenize_function, batched=True)
val_ds2_tok = val_ds2.map(tokenize_function, batched=True)

trainer2 = Trainer(
    args=training_args,
    model=model,
    train_dataset=new_train_ds_tok,
    eval_dataset=val_ds2_tok,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

trainer2.train()


Map:   0%|          | 0/249 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.303977,0.957771,0.800000
2,0.220734,0.619176,0.850000
3,0.011591,0.328726,0.850000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/fvcg/projects/uni-ai/.ai-env/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/fvcg/projects/uni-ai/.ai-env/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=96, training_loss=0.1689281592456003, metrics={'train_runtime': 870.5806, 'train_samples_per_second': 0.858, 'train_steps_per_second': 0.11, 'total_flos': 98953146796032.0, 'train_loss': 0.1689281592456003, 'epoch': 3.0})